In [62]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [63]:
MY_GENE = "DPYD"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on DPYD


In [64]:
# TODO Q1
#   (a) if MY_FILE exists, read it; otherwise fetch_gene(MY_GENE) and save it
#   (b) pass it through numeric() so scores are numbers, not strings
#   (c) add a `label` column using clinvar_label()
#   (d) run and print the three quality checks described above
#
# Everything you need is already defined. This is about eight lines.

# Setting up the cache file. Load the cache if it exists,
# otherwise we fetch the data from MyVariant.info and save.

# MyVariant.info is an API which collates variant data from
# multiple sources. 
CACHE = DATA / f"{MY_GENE}.tsv"
if CACHE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(CACHE, sep="\t")
    print(f"loaded cached {CACHE}")
else:
    print("fetching from MyVariant.info; this takes a few minutes\n")
    G =fetch_gene(MY_GENE)
    G["note"] = G.rsid.map(ANCHORS)
    G.to_csv(CACHE, sep="\t", index=False)
    print(f"\nsaved {CACHE}")

G = numeric(G) # force score columns to numbers
G["label"] = G.clinvar_sig.map(clinvar_label) # add a `label` column using clinvar_label()
G.to_csv(CACHE, sep="\t", index=False) # save the update version to the cache


# Three Checks:
# Number of variants:
print("Number of variants:", len(G))

# Known Star Alleles? 
print("Known star alleles:", G["note"].notna().sum())

# Are CADD, REVEL, and AlphaMissense scores populated?
print("CADD scores populated:", G["cadd_phred"].notna().sum())
print("REVEL scores populated:", G["revel"].notna().sum())
print("AlphaMissense scores populated:", G["alphamissense"].notna().sum())

# Everything looks good. 

loaded cached data\DPYD.tsv
Number of variants: 426
Known star alleles: 3
CADD scores populated: 304
REVEL scores populated: 161
AlphaMissense scores populated: 163


In [65]:
# TODO Q2
#   (a) group G by consequence and count non-null scores per tool
#   (b) print the table, plus the overall percentage scored by each tool
#   (c) print which consequence classes have ZERO REVEL scores
#
# Hint: .groupby("consequence") then .apply(lambda s: s.notna().sum())

TOOLS = ["cadd_phred", "revel", "alphamissense"]

grouped_g = G.groupby("consequence") # Group the returned MyVariant.info data by consequence
coverage_table = pd.DataFrame({"n": grouped_g.size()}) # create a new DF for the coverage table
for t in TOOLS:
    coverage_table[t] = grouped_g[t].apply(lambda s: s.notna().sum()) # count non-null scores per tool
coverage_table = coverage_table.sort_values("n", ascending=False)
coverage_table.loc["TOTAL"] = coverage_table.sum()

print("number of variants each tool returned a score for\n")
print(coverage_table.to_string())
print("\npercent of the whole variant set scored by each tool")
print((coverage_table.loc["TOTAL", TOOLS] / coverage_table.loc["TOTAL", "n"] * 100).round(1).to_string())

# Of the 400 variants, CADD was able to score 76%, across various consequence classes.
# Revel and AlphaMissense scored compartiavely worse at 40.2% and 40.8% respectively, 
# and were only able to score missense variants.

# Interestingly, of the 29 frameshift variants, none of the 3 tools were able to score them.


number of variants each tool returned a score for

                           n  cadd_phred  revel  alphamissense
consequence                                                   
missense_variant         163         161    161            163
intron_variant            53          44      0              0
synonymous_variant        48          47      0              0
3_prime_UTR_variant       34          11      0              0
stop_gained               30          29      0              0
frameshift_variant        29           0      0              0
splice_donor_variant      17           4      0              0
splice_acceptor_variant   16           2      0              0
splice_region_variant      7           4      0              0
upstream_gene_variant      3           2      0              0
TOTAL                    400         304    161            163

percent of the whole variant set scored by each tool
cadd_phred       76.0
revel            40.2
alphamissense    40.8


In [66]:
# TODO Q3
#   (a) set MAX_AF to a threshold you can defend for YOUR gene
#   (b) report how many variants it removes
#   (c) CRITICAL: list the P/LP or "drug response" variants your filter removes.
#       If it removes a known functional allele, either change the threshold or
#       justify keeping it in your report.
#   (d) for contrast, report what a 0.001 rare-disease filter would have removed
#   (e) set G to the filtered set

MAX_AF = 0.02

# gnomad_af is the allele frequency in the general population. 
# Common variants are unlikely to be pathogenic, so we can filter them out.
G["gnomad_af"] = pd.to_numeric(G["gnomad_af"], errors="coerce") # Make sure gnomad_af is numeric, so we can filter it
removed = G[G["gnomad_af"] > MAX_AF] # Make a list of the variants that are removed by the filter

print(f"Filtered out {len(removed)} variants with gnomad_af > {MAX_AF}")
important_variants_removed = removed[removed["label"].eq("P/LP")
    | removed["clinvar_sig"].str.contains("drug response", case=False, na=False)
]
print("P/LP or drug-response variants removed:")
print(important_variants_removed[["rsid", "clinvar_sig", "gnomad_af"]].to_string(index=False))

#MAX_AF = 0.001
#Filtered out 69 variants with gnomad_af > 0.001
#P/LP or drug-response variants removed:
#      rsid                  clinvar_sig  gnomad_af
# rs3918290 Pathogenic/Likely pathogenic   0.004555
#rs75017182                drug response   0.013111
# rs1801265                   Pathogenic   0.273227

# MAX_AF = 0.01
#Filtered out 50 variants with gnomad_af > 0.01
#P/LP or drug-response variants removed:
#      rsid   clinvar_sig  gnomad_af
#rs75017182 drug response   0.013111
# rs1801265    Pathogenic   0.273227


Filtered out 38 variants with gnomad_af > 0.02
P/LP or drug-response variants removed:
     rsid clinvar_sig  gnomad_af
rs1801265  Pathogenic   0.273227


In [67]:
# TODO Q4
#   (a) choose SCORE_COL and BANDS and run calibrate() on G
#   (b) print how many labelled variants the calibration rests on
#   (c) print how many bands rest on fewer than 10 variants
#   (d) if your gene is too small to calibrate, say so in the output and take
#       one of the two documented routes above
#
# Suggested bands: CADD [0,10,20,25,30,50]   REVEL [0,.2,.4,.6,.8,1.0]

SCORE_COL = "cadd_phred"  # I chose this for higher coverage across consequence classes.
BANDS     = [0, 10, 20, 25, 30, 50]

c = calibrate(G, SCORE_COL, BANDS)
print(f"\n{SCORE_COL} calibrated against ClinVar  "
        f"(P/LP n={c.attrs['n_p']}, B/LB n={c.attrs['n_b']})\n")
print(c.to_string())
print(f"\n{(c.n < 10).sum()} bands rest on fewer than 10 variants") # (25-30)

#cadd_phred calibrated against ClinVar  (P/LP n=124, B/LB n=160)
#
#            n  n_PLP  n_BLB   LR+                evidence
#band                                                     
#(0, 10]   117      0    117  0.00          strong, benign
#(10, 20]   36      0     36  0.00          strong, benign
#(20, 25]   11      5      6  1.08  no evidence either way
#(25, 30]    4      3      1  3.87  supporting, pathogenic
#(30, 50]  116    116      0   inf      strong, pathogenic

# The 25-30 band rests on only 4 variants, so the LR+ is not reliable. 
# The other bands are are well supported by the evidence and align
# with what is expected. 


cadd_phred calibrated against ClinVar  (P/LP n=39, B/LB n=97)

           n  n_PLP  n_BLB   LR+                evidence
band                                                    
(0, 10]   66      0     66  0.00          strong, benign
(10, 20]  24      0     24  0.00          strong, benign
(20, 25]  11      5      6  2.07  no evidence either way
(25, 30]   4      3      1  7.46    moderate, pathogenic
(30, 50]  31     31      0   inf      strong, pathogenic

1 bands rest on fewer than 10 variants


In [68]:
# TODO Q5
#   (a) take the VUS rows from G
#   (b) map each one to the evidence strength of its calibrated band
#   (c) count how many move towards pathogenic, towards benign, and stay put
#   (d) export to pgx_classified_<MY_GENE>.csv with these columns:
#       rsid, gene, consequence, aa_change, your score column, gnomad_af,
#       evidence, provisional_call

vus = G[G["label"] == "VUS"].copy() # VUS columns are variants of uncertain signifigance. 
vus[SCORE_COL] = pd.to_numeric(vus[SCORE_COL], errors="coerce")

vus["band"] = pd.cut(vus[SCORE_COL], bins=BANDS)
vus["evidence"] = vus["band"].map(c["evidence"])

vus["provisional_call"] = "stay put"
vus.loc[vus["evidence"].str.contains("pathogenic", na=False), "provisional_call"] = "towards pathogenic"
vus.loc[vus["evidence"].str.contains("benign", na=False), "provisional_call"] = "towards benign"
vus.loc[vus["evidence"].isna(), "provisional_call"] = "not assessed"

# I am simply mapping the CADD score to a band, and then mapping that band to the evidence strength,
# based on what I found in the calibration step.

print(vus["provisional_call"].value_counts())

cols = [
    "rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af",
    "evidence", "provisional_call"
]
vus[cols].to_csv(f"pgx_classified_{MY_GENE}.csv", index=False)

provisional_call
towards benign        56
towards pathogenic    53
not assessed          38
stay put              36
Name: count, dtype: int64


In [69]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200

from dotenv import load_dotenv; load_dotenv() # I used a .env file to store my API key

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"hgvs_g": r.hgvs_g, "rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="hgvs_g", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["hgvs_g", "rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

  100/200 queried
  150/200 queried
  200/200 queried

scored 151 variants (0 failures)

highest-impact DPYD variants by AVI

            hgvs_g          consequence  avi_phred         top_feature  revel  cadd_phred label
chr1:g.97549592G>A          stop_gained      45.11 PROTEIN_TERMINATION    NaN        42.0  P/LP
chr1:g.97549676G>A          stop_gained      40.75 PROTEIN_TERMINATION    NaN        41.0  P/LP
chr1:g.97828142C>A          stop_gained      40.58 PROTEIN_TERMINATION    NaN        38.0  P/LP
chr1:g.97740448T>A          stop_gained      40.47 PROTEIN_TERMINATION    NaN        38.0 other
chr1:g.97595122G>A          stop_gained      40.17 PROTEIN_TERMINATION    NaN        35.0  P/LP
chr1:g.97699523G>A          stop_gained      40.12 PROTEIN_TERMINATION    NaN        38.0  P/LP
chr1:g.97573943C>A          stop_gained      39.43 PROTEIN_TERMINATION    NaN        42.0  P/LP
chr1:g.97691782G>A          stop_gained      39.01 PROTEIN_TERMINATION    NaN        39.0  P/LP
chr1:g.970

In [74]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

# Lets investigate these
cols = [col for col in [
    "hgvs_g", "rsid", "note", "consequence", "aa_change",
    "revel", "alphamissense", "label"
] if col in revel_vs_am.columns]

print(f"\nREVEL vs AlphaMissense discordant variants: {len(revel_vs_am)}")
print(revel_vs_am[cols].to_string(index=False))


flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")


REVEL vs AlphaMissense discordant variants: 35
            hgvs_g           note      consequence aa_change  revel  alphamissense       label
chr1:g.97450189C>T            NaN missense_variant     R592Q  0.822         0.1896         VUS
chr1:g.97373596T>C            NaN missense_variant     M675V  0.881         0.4766         VUS
chr1:g.97079056C>T            NaN missense_variant    D1000N  0.629         0.1301         VUS
chr1:g.97699489T>G            NaN missense_variant     K181T  0.540         0.0812       other
chr1:g.97883268C>A            NaN missense_variant      C49F  0.619         0.1732         VUS
chr1:g.97699486A>T            NaN missense_variant     M182K  0.721         0.1696         VUS
chr1:g.97593274G>A            NaN missense_variant     R358C  0.747         0.2788        P/LP
chr1:g.97079002G>C            NaN missense_variant    P1018A  0.618         0.0757         VUS
chr1:g.97679170T>C            NaN missense_variant     K259E  0.611         0.1137        B/LB
ch